In [4]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import LabelEncoder

# ==============================
# 1. DATA GENERATION
# ==============================
def generate_realistic_data(n=5000):
    np.random.seed(42)

    genres = [
        'Action','Comedy','Drama','Horror','Sci-Fi',
        'Romance','Thriller','Fantasy','Adventure',
        'Mystery','Crime','Biography','History','War',
        'Animation','Family','Sport','Musical','Documentary','Western'
    ]

    stability = ['Drop Fast', 'Stable', 'Growing']

    df = pd.DataFrame({
        'Genre': np.random.choice(genres, n),

        'Budget': np.random.randint(10, 500, n),
        'Marketing_Budget': np.random.randint(5, 200, n),

        'Screens': np.random.randint(200, 30000, n),
        'Weeks': np.random.randint(2, 12, n),
        'Shows': np.random.randint(2, 15, n),

        'Ticket_Price': np.random.randint(100, 800, n),

        'Screen_Stability': np.random.choice(stability, n),

        'Actor_Popularity': np.random.randint(1, 10, n),
        'Director_Success': np.random.randint(1, 10, n),

        'Occupancy': np.random.uniform(0.25, 0.95, n),

        #  NEW FEATURE (IMPORTANT)
        'Seats_Per_Screen': np.random.randint(80, 250, n)
    })

    # ==============================
    #  BALANCED SIGNAL (NO FORMULA)
    # ==============================
    score = (
        df['Occupancy'] * 6 +
        df['Seats_Per_Screen'] * 0.03 +   #  NEW IMPACT
        df['Screens'] * 0.00015 +
        df['Shows'] * 0.25 +
        df['Weeks'] * 0.6 +
        df['Actor_Popularity'] * 0.4 +
        df['Director_Success'] * 0.35 +
        df['Marketing_Budget'] * 0.01 +
        df['Budget'] * (-0.0035) +
        np.random.normal(0, 1.0, n)
    )

    threshold = np.percentile(score, 60)
    df['Target'] = (score > threshold).astype(int)

    df.to_csv('final_dataset_inr.csv', index=False)

    return df


# ==============================
# 2. TRAIN MODEL
# ==============================
def train_model():
    df = generate_realistic_data(5000)

    hit_df = df[df['Target'] == 1]
    flop_df = df[df['Target'] == 0]

    flop_df = flop_df.sample(min(len(flop_df), len(hit_df)*2), random_state=42)
    df = pd.concat([hit_df, flop_df])

    print("\n Target Distribution:")
    print(df['Target'].value_counts())

    # Encode
    encoders = {}
    for col in ['Genre', 'Screen_Stability']:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        encoders[col] = le

    X = df.drop(['Target'], axis=1)
    y = df['Target']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    model = RandomForestClassifier(
        n_estimators=400,
        class_weight={0:1, 1:2},
        random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print("\n MODEL PERFORMANCE")
    print(f"Accuracy:  {accuracy_score(y_test, y_pred)*100:.2f}%")
    print(f"Precision: {precision_score(y_test, y_pred):.2f}")
    print(f"Recall:    {recall_score(y_test, y_pred):.2f}")
    print(f"F1 Score:  {f1_score(y_test, y_pred):.2f}")

    print("\nDetailed Report:")
    print(classification_report(y_test, y_pred))

    importance = pd.Series(model.feature_importances_, index=X.columns)
    print("\n  Feature Importance:")
    print(importance.sort_values(ascending=False))

    joblib.dump(model, "model_inr.pkl")
    joblib.dump(encoders, "encoders_inr.pkl")

    print("\n FINAL MODEL READY ")


# ==============================
# RUN
# ==============================
if __name__ == "__main__":
    train_model()


 Target Distribution:
Target
0    3000
1    2000
Name: count, dtype: int64

 MODEL PERFORMANCE
Accuracy:  85.40%
Precision: 0.88
Recall:    0.73
F1 Score:  0.80

Detailed Report:
              precision    recall  f1-score   support

           0       0.84      0.93      0.88       600
           1       0.88      0.73      0.80       400

    accuracy                           0.85      1000
   macro avg       0.86      0.83      0.84      1000
weighted avg       0.86      0.85      0.85      1000


  Feature Importance:
Weeks               0.165014
Seats_Per_Screen    0.150076
Occupancy           0.142907
Screens             0.131547
Actor_Popularity    0.073030
Marketing_Budget    0.064047
Shows               0.061139
Budget              0.059974
Director_Success    0.054680
Ticket_Price        0.048928
Genre               0.035879
Screen_Stability    0.012778
dtype: float64

 FINAL MODEL READY 
